# LoRA: Training-Label Noise Experiments

**MIA5150 — BERT with LoRA adapters and a classification head**  
**Team:** Callum Anderson and Zaynab Othman Taha

Supervised 11-class paper classification using the same pretrained BERT encoder and raw [CLS] → linear head as frozen and full fine-tuning. Train LoRA adapters and the head while keeping the original encoder weights frozen. Each rank/seed/noise condition starts from the same pretrained base.

## Imports

If PEFT is missing in Colab, run `%pip install peft` in a temporary code cell, restart the session if requested, then run this notebook from the top.

The imports cell checks PEFT’s optional TorchAO backend. If Colab contains an incompatible TorchAO version, it removes that unused package from the current runtime. This experiment uses float32 LoRA, so TorchAO is not needed. PyTorch and CUDA are left unchanged.


In [14]:
import gc
import importlib
import subprocess
import sys
import hashlib
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import peft
from peft import (
    LoraConfig, TaskType, get_peft_model,
    get_peft_model_state_dict, set_peft_model_state_dict,
)
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import transformers
from transformers import AutoModel, AutoTokenizer, BertConfig, set_seed

from IPython.display import display

# Colab may include an outdated optional quantization backend.
# This float32 experiment does not use TorchAO; leave PyTorch/CUDA unchanged.
from peft.import_utils import is_torchao_available

try:
    is_torchao_available()
except ImportError as dependency_error:
    if "incompatible version of torchao" not in str(dependency_error):
        raise
    print(f"PEFT compatibility fix: {dependency_error}")
    print("Removing unused TorchAO from this runtime; PyTorch and CUDA are unchanged.")
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
    importlib.invalidate_caches()
    is_torchao_available.cache_clear()
    if is_torchao_available():
        raise RuntimeError("TorchAO is still detected. Restart the kernel and rerun the notebook.")
    print("Compatibility fix complete. Continue with the notebook.")


PEFT compatibility fix: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported
Removing unused TorchAO from this runtime; PyTorch and CUDA are unchanged.
Compatibility fix complete. Continue with the notebook.


# 1. Verify GPU, as it will be needed for this experiment

In [15]:
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


# 2. Load the prepared data and set the experiment conditions

The conditions are **0%, 10%, 20%, and 40% added training-label noise**, with
**seeds 42, 43, and 44**. 

In [16]:
LABEL_COLUMN = "label (target)"
NUMBER_OF_CLASSES = 11
CLASS_IDS = np.arange(NUMBER_OF_CLASSES)
NOISE_PERCENTAGES = (0, 10, 20, 40)
RANDOM_SEEDS = (42, 43, 44)
SPLIT_NAMES = ("train", "validation", "test")

DATA_DIRECTORY = Path("/content")
source_paths = {split: DATA_DIRECTORY / f"{split}_merged.csv" for split in SPLIT_NAMES}
missing_files = [path.name for path in source_paths.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing files in /content: {', '.join(missing_files)}. "
        "Upload the three cleaned CSVs directly to the Colab Files panel."
    )

source_hashes = {
    split: hashlib.sha256(path.read_bytes()).hexdigest()
    for split, path in source_paths.items()
}
data_splits = {
    split: pd.read_csv(path, dtype={"paper_id": "string"})
    for split, path in source_paths.items()
}
for split, frame in data_splits.items():
    assert frame["paper_id"].notna().all() and frame["paper_id"].is_unique, split
    assert pd.api.types.is_integer_dtype(frame[LABEL_COLUMN]), split
    assert set(frame[LABEL_COLUMN]) == set(CLASS_IDS), split
    for text_column in ("title", "abstract"):
        assert frame[text_column].fillna("").str.strip().ne("").all(), split

for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert set(data_splits[left]["paper_id"]).isdisjoint(data_splits[right]["paper_id"])

training_data = data_splits["train"]
validation_data = data_splits["validation"]
test_data = data_splits["test"]
original_training_labels = training_data[LABEL_COLUMN].to_numpy(dtype=np.int64, copy=True)
original_training_labels.setflags(write=False)
original_split_labels = {split: frame[LABEL_COLUMN].copy(deep=True) for split, frame in data_splits.items()}
class_names = training_data.groupby(LABEL_COLUMN)["classname"].first().reindex(CLASS_IDS)
for frame in data_splits.values():
    assert frame["classname"].eq(frame[LABEL_COLUMN].map(class_names)).all()

print(f"Loaded {sum(len(frame) for frame in data_splits.values()):,} unique papers from {DATA_DIRECTORY}.")


Loaded 23,335 unique papers from /content.


# 3. Add symmetric training-label noise

For each condition, select records **proportionally from each original training class**
and replace each selected label uniformly with **one of the other ten classes**.
Every selected label therefore changes. Validation and test labels stay unchanged.


In [17]:
def add_symmetric_label_noise(original_labels, noise_percent, seed):
    """Return independent noisy labels and a mask of changed records."""
    labels = np.asarray(original_labels)
    if labels.ndim != 1 or labels.size == 0 or not np.issubdtype(labels.dtype, np.integer):
        raise ValueError("Labels must be a nonempty one-dimensional integer array.")
    if not np.isin(labels, CLASS_IDS).all():
        raise ValueError("Labels must be integers from 0 to 10.")
    if not isinstance(noise_percent, (int, np.integer)) or not 0 <= noise_percent <= 100:
        raise ValueError("noise_percent must be an integer between 0 and 100.")
    if not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError("seed must be a nonnegative integer.")

    noisy_labels = labels.copy()
    changed_mask = np.zeros(len(labels), dtype=bool)
    original_class_counts = np.bincount(labels, minlength=NUMBER_OF_CLASSES)
    quota_numerators = original_class_counts * noise_percent
    flips_per_class = quota_numerators // 100
    target_flips = (len(labels) * noise_percent + 50) // 100
    remaining = int(target_flips - flips_per_class.sum())
    remainder_order = np.argsort(-(quota_numerators % 100), kind="stable")
    flips_per_class[remainder_order[:remaining]] += 1

    rng = np.random.default_rng(np.random.SeedSequence([int(seed), int(noise_percent)]))
    for class_id, number_to_flip in enumerate(flips_per_class):
        if number_to_flip == 0:
            continue
        eligible_positions = np.flatnonzero(labels == class_id)
        chosen_positions = rng.choice(eligible_positions, size=int(number_to_flip), replace=False)
        alternative_labels = CLASS_IDS[CLASS_IDS != class_id]
        noisy_labels[chosen_positions] = rng.choice(alternative_labels, size=int(number_to_flip))
        changed_mask[chosen_positions] = True

    assert changed_mask.sum() == target_flips
    assert np.array_equal(changed_mask, noisy_labels != labels)
    return noisy_labels, changed_mask


## 3.1 Generate the reusable label variants

`training_label_variants[(seed, noise_percent)]` holds the labels aligned with
`training_data`. That same entry is used for frozen BERT, LoRA, and full fine-tuning.



In [18]:
training_label_variants = {}
noise_audits = {}

for seed in RANDOM_SEEDS:
    for noise_percent in NOISE_PERCENTAGES:
        noisy_labels, changed_mask = add_symmetric_label_noise(
            original_training_labels, noise_percent, seed
        )
        noisy_labels.setflags(write=False)
        key = (seed, noise_percent)
        training_label_variants[key] = noisy_labels
        noise_audits[key] = pd.DataFrame({
            "paper_id": training_data["paper_id"].to_numpy(copy=True),
            "original_label": original_training_labels.copy(),
            "noisy_label": noisy_labels.copy(),
            "label_changed": changed_mask.copy(),
        })


## 3.2 Verify the scheme

Check exact flip totals, proportional allocation, valid replacement labels,
reproducibility, and alignment to paper IDs. Also confirm the original labels and
all three source files are unchanged. These checks do not establish that the
original disease labels are semantically correct.

In [19]:
original_class_counts = np.bincount(original_training_labels, minlength=NUMBER_OF_CLASSES)
for (seed, noise_percent), noisy_labels in training_label_variants.items():
    audit = noise_audits[(seed, noise_percent)]
    changed = noisy_labels != original_training_labels
    expected_flips = (len(training_data) * noise_percent + 50) // 100
    assert int(changed.sum()) == expected_flips
    assert np.isin(noisy_labels, CLASS_IDS).all()
    assert not np.shares_memory(noisy_labels, original_training_labels)
    assert np.array_equal(audit["paper_id"], training_data["paper_id"])
    assert np.array_equal(audit["original_label"], original_training_labels)
    assert np.array_equal(audit["noisy_label"], noisy_labels)
    assert np.array_equal(audit["label_changed"], changed)

    actual_per_class = np.bincount(original_training_labels[changed], minlength=NUMBER_OF_CLASSES)
    lower_quota = original_class_counts * noise_percent // 100
    upper_quota = (original_class_counts * noise_percent + 99) // 100
    assert ((actual_per_class >= lower_quota) & (actual_per_class <= upper_quota)).all()
    repeated_labels, repeated_mask = add_symmetric_label_noise(
        original_training_labels, noise_percent, seed
    )
    assert np.array_equal(noisy_labels, repeated_labels)
    assert np.array_equal(changed, repeated_mask)

for split, frame in data_splits.items():
    pd.testing.assert_series_equal(frame[LABEL_COLUMN], original_split_labels[split])
    assert hashlib.sha256(source_paths[split].read_bytes()).hexdigest() == source_hashes[split]
assert np.array_equal(training_data[LABEL_COLUMN], original_training_labels)
print("Passed: all 12 variants are reproducible, correctly allocated, and aligned.")
print("Original training, validation, test labels and CSV files remain unchanged.")


Passed: all 12 variants are reproducible, correctly allocated, and aligned.
Original training, validation, test labels and CSV files remain unchanged.


# 4. Tokenize titles and abstracts

BERT-base-uncased tokenizer and 256-token limit.


In [20]:
BERT_CHECKPOINT_NAME = "google-bert/bert-base-uncased"
MAXIMUM_TOKEN_LENGTH = 256

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_CHECKPOINT_NAME, use_fast=True)


def tokenize_papers(paper_data):
    return bert_tokenizer(
        text=paper_data["title"].tolist(),
        text_pair=paper_data["abstract"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAXIMUM_TOKEN_LENGTH,
        return_tensors="pt",
    )


tokenized_splits = {
    split: tokenize_papers(data_splits[split])
    for split in SPLIT_NAMES
}
training_tokenized_inputs = tokenized_splits["train"]
validation_tokenized_inputs = tokenized_splits["validation"]
test_tokenized_inputs = tokenized_splits["test"]

print("Tokenizer:", BERT_CHECKPOINT_NAME)
print("Maximum sequence length:", MAXIMUM_TOKEN_LENGTH)
for split, encoded_inputs in tokenized_splits.items():
    expected_shape = (len(data_splits[split]), MAXIMUM_TOKEN_LENGTH)
    for input_name in ("input_ids", "attention_mask", "token_type_ids"):
        assert tuple(encoded_inputs[input_name].shape) == expected_shape, (split, input_name)
    print(f"{split}: {expected_shape[0]:,} records × {expected_shape[1]} tokens")


Tokenizer: google-bert/bert-base-uncased
Maximum sequence length: 256
train: 16,334 records × 256 tokens
validation: 3,499 records × 256 tokens
test: 3,502 records × 256 tokens


# 5. Prepare BERT with LoRA

Use the same **google-bert/bert-base-uncased** encoder and raw **[CLS] → linear 11-class classifier**. Omit the unused pooler. Add standard LoRA to the attention **query and value projections in every BERT layer**; freeze the original encoder weights and biases. Train only the adapters and the external linear head.


In [21]:
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime, then rerun the notebook before training.")
computation_device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(computation_device))
TOKEN_INPUT_NAMES = ("input_ids", "attention_mask", "token_type_ids")

class LoRABertClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.bert = encoder
        self.classifier = nn.Linear(encoder.config.hidden_size, NUMBER_OF_CLASSES)

    def forward(self, **tokenized_inputs):
        encoded = self.bert(**tokenized_inputs)
        return self.classifier(encoded.last_hidden_state[:, 0, :])

def prepare_pretrained_base():
    global lora_base_state, lora_base_config
    encoder = AutoModel.from_pretrained(BERT_CHECKPOINT_NAME, add_pooling_layer=False).float()
    lora_base_config = encoder.config.to_dict()
    lora_base_state = {name: value.detach().cpu().clone()
                       for name, value in encoder.state_dict().items()}
    del encoder
    gc.collect()

def create_lora_model(rank, seed):
    encoder = AutoModel.from_config(BertConfig.from_dict(lora_base_config), add_pooling_layer=False)
    # Restore pretrained weights before installing adapters; never train a random base.
    encoder.load_state_dict(lora_base_state)
    configuration = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION, r=rank, lora_alpha=LORA_ALPHA_MULTIPLIER * rank,
        lora_dropout=LORA_DROPOUT, target_modules=list(LORA_TARGET_MODULES),
        bias="none", init_lora_weights=True, use_rslora=False,
    )
    set_seed(seed)
    encoder = get_peft_model(encoder, configuration)
    # Match the seeded external nn.Linear head used by frozen and full fine-tuning.
    set_seed(seed)
    model = LoRABertClassifier(encoder).float().to(computation_device)
    adapted_modules = [name for name, module in model.bert.named_modules() if hasattr(module, "lora_A")]
    assert len(adapted_modules) == 2 * model.bert.config.num_hidden_layers
    assert all(name.rsplit(".", 1)[-1] in LORA_TARGET_MODULES for name in adapted_modules)
    assert all(parameter.requires_grad == ("lora_" in name)
               for name, parameter in model.bert.named_parameters())
    assert all(parameter.requires_grad for parameter in model.classifier.parameters())
    expected_trainable = (4 * rank * model.bert.config.hidden_size * model.bert.config.num_hidden_layers
                          + (model.bert.config.hidden_size + 1) * NUMBER_OF_CLASSES)
    assert sum(p.numel() for p in model.parameters() if p.requires_grad) == expected_trainable
    return model

def synchronize_gpu():
    if computation_device.type == "cuda":
        torch.cuda.synchronize(computation_device)

def reset_gpu_peak_memory():
    if computation_device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(computation_device)

def peak_gpu_memory_gib():
    return (torch.cuda.max_memory_allocated(computation_device) / 1024 ** 3
            if computation_device.type == "cuda" else np.nan)

def maximum_measured_memory(*values):
    measured = [float(value) for value in values if np.isfinite(value)]
    return max(measured) if measured else np.nan

def make_batches(split, labels, batch_size, shuffle=False, generator=None):
    inputs = tokenized_splits[split]
    dataset = TensorDataset(*(inputs[name] for name in TOKEN_INPUT_NAMES),
                            torch.tensor(np.asarray(labels).copy(), dtype=torch.long))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      pin_memory=computation_device.type == "cuda", generator=generator)

def unpack_batch(batch):
    inputs = {name: values.to(computation_device, non_blocking=True)
              for name, values in zip(TOKEN_INPUT_NAMES, batch[:-1])}
    return inputs, batch[-1].to(computation_device, non_blocking=True)

@torch.no_grad()
def evaluate_lora_model(model, split, true_labels):
    model.eval()
    batches = make_batches(split, true_labels, EVALUATION_BATCH_SIZE,
                           generator=torch.Generator().manual_seed(0))
    predictions = []
    for batch in batches:
        inputs, _ = unpack_batch(batch)
        predictions.append(model(**inputs).argmax(dim=1).cpu())
    predictions = torch.cat(predictions).numpy()
    return {
        "accuracy": float(accuracy_score(true_labels, predictions)),
        "macro_f1": float(f1_score(true_labels, predictions, labels=CLASS_IDS,
                                  average="macro", zero_division=0)),
        "predictions": predictions,
    }

true_validation_labels = validation_data[LABEL_COLUMN].to_numpy(dtype=np.int64, copy=True)
true_test_labels = test_data[LABEL_COLUMN].to_numpy(dtype=np.int64, copy=True)

def restore_lora_model(run_key):
    """Rebuild a final model from the shared base plus its adapters and head."""
    rank, seed, noise_percent = run_key
    metadata = lora_run_metadata[run_key]
    assert lora_training_config == current_training_config(), "Training settings changed; rerun section 6."
    assert metadata["training_run_id"] == lora_training_run_id == lora_base_training_run_id
    assert metadata["epoch"] == NUMBER_OF_EPOCHS, "Finish all fixed epochs before evaluation."
    assert metadata["noise_label_sha256"] == hashlib.sha256(
        training_label_variants[(seed, noise_percent)].astype(np.int64).tobytes()
    ).hexdigest(), "Training labels changed; rerun section 6."
    model = create_lora_model(rank, seed)
    adapter_result = set_peft_model_state_dict(model.bert, lora_model_states[run_key]["adapter"])
    assert not adapter_result.unexpected_keys, adapter_result.unexpected_keys
    assert not any("lora_" in name for name in adapter_result.missing_keys), adapter_result.missing_keys
    model.classifier.load_state_dict(lora_model_states[run_key]["head"])
    return model, metadata


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


# 6. Train the LoRA adapters and classification head

Cross-entropy loss, AdamW, weight decay 0.01, training batch size 32, and **10 fixed epochs**, matching frozen and full fine-tuning. Use **0.00002** as the controlled learning rate shared with full fine-tuning, fixed across all LoRA ranks, noise levels, and seeds. 


In [22]:
NUMBER_OF_EPOCHS = 10
TRAINING_BATCH_SIZE = 32
EVALUATION_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
LORA_RANKS = (8,)
LORA_ALPHA_MULTIPLIER = 2
LORA_DROPOUT = 0.0
LORA_TARGET_MODULES = ("query", "value")
assert LORA_RANKS and len(set(LORA_RANKS)) == len(LORA_RANKS)
assert all(isinstance(rank, int) and rank > 0 for rank in LORA_RANKS)

def current_training_config():
    return {
        "method": "lora", "checkpoint": BERT_CHECKPOINT_NAME,
        "readout": "raw_cls_linear", "precision": "float32",
        "max_length": MAXIMUM_TOKEN_LENGTH, "source_hashes": dict(source_hashes),
        "class_names": class_names.tolist(), "epochs": NUMBER_OF_EPOCHS,
        "batch_size": TRAINING_BATCH_SIZE, "evaluation_batch_size": EVALUATION_BATCH_SIZE,
        "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "stopping_rule": "final_epoch", "ranks": list(LORA_RANKS),
        "alpha_multiplier": LORA_ALPHA_MULTIPLIER, "lora_dropout": LORA_DROPOUT,
        "target_modules": list(LORA_TARGET_MODULES), "bias": "none",
        "init_lora_weights": True, "use_rslora": False, "task_type": "FEATURE_EXTRACTION",
        "seeds": list(RANDOM_SEEDS), "noise_percentages": list(NOISE_PERCENTAGES),
    }

def current_environment():
    return {"torch": str(torch.__version__), "transformers": str(transformers.__version__),
            "peft": str(peft.__version__),
            "gpu": torch.cuda.get_device_name(computation_device) if computation_device.type == "cuda" else "CPU"}

def check_source_data():
    for split in SPLIT_NAMES:
        assert hashlib.sha256(source_paths[split].read_bytes()).hexdigest() == source_hashes[split]
        pd.testing.assert_series_equal(data_splits[split][LABEL_COLUMN], original_split_labels[split])

print(f"Runs: {len(LORA_RANKS) * len(RANDOM_SEEDS) * len(NOISE_PERCENTAGES)}; LoRA ranks: {LORA_RANKS}")
print(f"Stopping rule: final epoch ({NUMBER_OF_EPOCHS})")
print(f"Learning rate: {LEARNING_RATE}; training batch size: {TRAINING_BATCH_SIZE}")


Runs: 12; LoRA ranks: (8,)
Stopping rule: final epoch (10)
Learning rate: 2e-05; training batch size: 32


## 6.1 Run the training conditions

For each rank/seed/noise combination, copy the original pretrained base, initialize fresh adapters and a fresh head, train on its assigned labels, and check the unchanged validation set after each epoch. Only adapters and the head enter the optimizer. The first batch verifies their gradients and confirms that frozen parameters receive none.

Keep one shared pretrained base and each run's final **adapter and head weights in CPU memory** for sections 7–8. Training writes no model files and does not resume interrupted runs; rerunning this cell trains every selected condition from scratch. Restarting or deleting the runtime loses these weights. Model initialization and copying final weights to CPU are excluded from training time.


In [23]:
check_source_data()
lora_training_config = current_training_config()
lora_training_run_id = time.time_ns()
lora_model_states = {}
lora_run_metadata = {}
lora_training_history_rows = []
prepare_pretrained_base()
lora_base_training_run_id = lora_training_run_id

for rank in LORA_RANKS:
    for seed in RANDOM_SEEDS:
        for noise_percent in NOISE_PERCENTAGES:
            run_key = (rank, seed, noise_percent)
            label_key = (seed, noise_percent)
            model = create_lora_model(rank, seed)
            trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
            optimizer = torch.optim.AdamW(trainable_parameters, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            loss_function = nn.CrossEntropyLoss()
            shuffle_generator = torch.Generator().manual_seed(seed)
            training_batches = make_batches("train", training_label_variants[label_key], TRAINING_BATCH_SIZE,
                                            shuffle=True, generator=shuffle_generator)
            history = []
            measurements = {"training_seconds": 0.0, "epoch_validation_seconds": 0.0,
                            "training_and_validation_seconds": 0.0, "peak_pytorch_allocated_gib": np.nan}
            parameter_count = sum(parameter.numel() for parameter in trainable_parameters)
            print(f"\nRank {rank}, seed {seed}, noise {noise_percent}% — {parameter_count:,} trainable parameters")
            synchronize_gpu()
            reset_gpu_peak_memory()
            for epoch in range(1, NUMBER_OF_EPOCHS + 1):
                model.train()
                synchronize_gpu()
                training_start = time.perf_counter()
                total_training_loss = 0.0
                for batch_index, batch in enumerate(tqdm(training_batches, desc=f"Epoch {epoch}/{NUMBER_OF_EPOCHS}", leave=False)):
                    inputs, targets = unpack_batch(batch)
                    optimizer.zero_grad(set_to_none=True)
                    logits = model(**inputs)
                    loss = loss_function(logits, targets)
                    if not torch.isfinite(loss):
                        raise FloatingPointError(f"Non-finite loss in rank {rank}, seed {seed}, noise {noise_percent}%, epoch {epoch}.")
                    loss.backward()
                    if epoch == 1 and batch_index == 0:
                        assert all(p.grad is not None for p in model.parameters() if p.requires_grad), "A trainable parameter received no gradient."
                        assert all(p.grad is None for p in model.parameters() if not p.requires_grad), "A frozen parameter received a gradient."
                    optimizer.step()
                    total_training_loss += loss.item() * len(targets)
                del inputs, targets, logits, loss, batch
                synchronize_gpu()
                measurements["training_seconds"] += time.perf_counter() - training_start
                average_loss = total_training_loss / len(training_data)
                validation_start = time.perf_counter()
                validation_metrics = evaluate_lora_model(model, "validation", true_validation_labels)
                synchronize_gpu()
                measurements["epoch_validation_seconds"] += time.perf_counter() - validation_start
                measurements["training_and_validation_seconds"] = measurements["training_seconds"] + measurements["epoch_validation_seconds"]
                measurements["peak_pytorch_allocated_gib"] = maximum_measured_memory(
                    measurements["peak_pytorch_allocated_gib"], peak_gpu_memory_gib())
                history.append({"lora_rank": rank, "seed": seed, "noise_percent": noise_percent, "epoch": epoch,
                                "training_loss": average_loss, "validation_accuracy": validation_metrics["accuracy"],
                                "validation_macro_f1": validation_metrics["macro_f1"]})
                print(f"Epoch {epoch:2d}/{NUMBER_OF_EPOCHS}: loss = {average_loss:.4f}, "
                      f"validation macro-F1 = {validation_metrics['macro_f1']:.4f}")

            lora_model_states[run_key] = {
                "adapter": {name: value.detach().cpu().clone()
                            for name, value in get_peft_model_state_dict(model.bert).items()},
                "head": {name: value.detach().cpu().clone()
                         for name, value in model.classifier.state_dict().items()},
            }
            lora_run_metadata[run_key] = {
                "training_run_id": lora_training_run_id, "epoch": epoch,
                "noise_label_sha256": hashlib.sha256(training_label_variants[label_key].astype(np.int64).tobytes()).hexdigest(),
                "measurements": dict(measurements), "trainable_parameters": parameter_count,
                "environment": current_environment(), "rank": rank, "lora_alpha": LORA_ALPHA_MULTIPLIER * rank,
            }
            lora_training_history_rows.extend(history)
            print(f"Training + epoch validation: {measurements['training_and_validation_seconds']:.1f} seconds")
            del model, optimizer, trainable_parameters, training_batches, validation_metrics
            gc.collect()
            if computation_device.type == "cuda":
                torch.cuda.empty_cache()

lora_training_history = pd.DataFrame(lora_training_history_rows)
assert set(lora_model_states) == set(lora_run_metadata) == {
    (rank, seed, noise) for rank in LORA_RANKS for seed in RANDOM_SEEDS for noise in NOISE_PERCENTAGES}
print("\nAll final-epoch adapters, heads, and epoch histories are held in memory. Continue to validation and final test evaluation.")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Rank 8, seed 42, noise 0% — 303,371 trainable parameters


Epoch 1/10:   0%|          | 0/511 [00:00<?, ?it/s]

Epoch  1/10: loss = 2.1375, validation macro-F1 = 0.1080


...

Epoch  8/10: loss = 1.7608, validation macro-F1 = 0.8553


Epoch 9/10:   0%|          | 0/511 [00:00<?, ?it/s]

Epoch  9/10: loss = 1.7500, validation macro-F1 = 0.8597


Epoch 10/10:   0%|          | 0/511 [00:00<?, ?it/s]

Epoch 10/10: loss = 1.7441, validation macro-F1 = 0.8599
Training + epoch validation: 429.0 seconds

All final-epoch adapters, heads, and epoch histories are held in memory. Continue to validation and final test evaluation.


# 7. Evaluate on validation

Rebuild each final-epoch model from the shared pretrained base, its adapters, and its head. Report individual accuracy and macro-F1 on unchanged validation labels. This does not change training settings or select the best epoch. The test set remains reserved for section 8.


In [24]:
check_source_data()
assert lora_training_config == current_training_config(), "Training settings changed; rerun section 6."
lora_validation_rows = []
lora_classification_reports = {}
lora_validation_predictions = {}
lora_validation_metadata = {}

for rank in LORA_RANKS:
    for seed in RANDOM_SEEDS:
        for noise_percent in NOISE_PERCENTAGES:
            run_key = (rank, seed, noise_percent)
            synchronize_gpu()
            reset_gpu_peak_memory()
            model, metadata = restore_lora_model(run_key)
            synchronize_gpu()
            validation_start = time.perf_counter()
            metrics = evaluate_lora_model(model, "validation", true_validation_labels)
            synchronize_gpu()
            validation_seconds = time.perf_counter() - validation_start
            lora_validation_predictions[run_key] = metrics["predictions"]
            lora_classification_reports[run_key] = classification_report(
                true_validation_labels, metrics["predictions"], labels=CLASS_IDS,
                target_names=class_names.tolist(), output_dict=True, zero_division=0)
            lora_validation_metadata[run_key] = {
                "training_run_id": metadata["training_run_id"], "seconds": validation_seconds,
                "peak_pytorch_allocated_gib": peak_gpu_memory_gib(),
            }
            lora_validation_rows.append({"lora_rank": rank, "seed": seed, "noise_percent": noise_percent,
                                        "final_epoch": metadata["epoch"], "accuracy": metrics["accuracy"],
                                        "macro_f1": metrics["macro_f1"],
                                        "training_seconds": metadata["measurements"]["training_seconds"]})
            del model
            gc.collect()
            if computation_device.type == "cuda":
                torch.cuda.empty_cache()

lora_validation_results = pd.DataFrame(lora_validation_rows)
print("Final-epoch validation results (accuracy and macro-F1 on a 0–1 scale):")
display(lora_validation_results.round(4))
lora_validation_training_run_id = lora_training_run_id


Final-epoch validation results (accuracy and macro-F1 on a 0–1 scale):


,lora_rank,seed,noise_percent,final_epoch,accuracy,macro_f1,training_seconds
0,8,42,0,10,0.8877,0.8842,389.7972
1,8,42,10,10,0.8828,0.8788,387.8587
2,8,42,20,10,0.8774,0.8720,387.9274
3,8,42,40,10,0.8645,0.8553,387.4582
4,8,43,0,10,0.8868,0.8791,387.3369
5,8,43,10,10,0.8820,0.8752,388.0948
6,8,43,20,10,0.8763,0.8689,387.0855
7,8,43,40,10,0.8654,0.8513,386.9612
8,8,44,0,10,0.8880,0.8827,387.6305
9,8,44,10,10,0.8811,0.8744,387.6293


In [25]:
REPORT_LORA_RANK = LORA_RANKS[0]
REPORT_SEED = 42
REPORT_NOISE_PERCENT = 0
report_key = (REPORT_LORA_RANK, REPORT_SEED, REPORT_NOISE_PERCENT)
print(f"Per-class validation scores: rank {REPORT_LORA_RANK}, seed {REPORT_SEED}, noise {REPORT_NOISE_PERCENT}%")
per_class_validation_report = pd.DataFrame(lora_classification_reports[report_key]).T.loc[
    class_names.tolist(), ["precision", "recall", "f1-score", "support"]]
per_class_validation_report["support"] = per_class_validation_report["support"].astype(int)
display(per_class_validation_report.round(4))


Per-class validation scores: rank 8, seed 42, noise 0%


,precision,recall,f1-score,support
Cardiovascular diseases,0.8038,0.9141,0.8554,780
Chronic kidney disease,0.8857,0.8532,0.8692,109
Chronic respiratory diseases,0.9156,0.8444,0.8785,257
Diabetes mellitus,0.9301,0.9142,0.9221,233
Digestive diseases,0.7339,0.7080,0.7207,113
HIV/AIDS,0.9290,0.9474,0.9381,152
Hepatitis A/B/C/E,1.0000,0.8852,0.9391,61
Mental disorders,0.9251,0.8843,0.9042,363
Musculoskeletal disorders,0.9113,0.9018,0.9065,285
Neoplasms (cancer),0.9361,0.8994,0.9174,815


# 8. Final evaluation on the held-out test set

Report each final-epoch run's accuracy, macro-F1, per-class precision/recall/F1 and support, training time, and peak **PyTorch-allocated GPU memory in GiB**.


In [26]:
expected_run_keys = {(rank, seed, noise) for rank in LORA_RANKS for seed in RANDOM_SEEDS for noise in NOISE_PERCENTAGES}
if "lora_validation_training_run_id" not in globals():
    raise RuntimeError("Run sections 5–7 before final test evaluation.")
assert lora_validation_training_run_id == lora_training_run_id, "Rerun section 7 for the current models."
assert lora_training_config == current_training_config(), "Training settings changed; rerun sections 6–7."
assert set(lora_model_states) == set(lora_run_metadata) == set(lora_validation_metadata) == expected_run_keys
validation_by_run = lora_validation_results.set_index(["lora_rank", "seed", "noise_percent"])
assert validation_by_run.index.is_unique and set(validation_by_run.index) == expected_run_keys
assert validation_by_run["final_epoch"].eq(NUMBER_OF_EPOCHS).all()
check_source_data()

RESULTS_PATH = DATA_DIRECTORY / "run_results.csv"
if RESULTS_PATH.is_file():
    run_registry = pd.read_csv(RESULTS_PATH, dtype=str, keep_default_na=False)
else:
    print("No existing results CSV found: creating a new registry without previous frozen or full-fine-tuning results. Upload the combined run_results.csv first to retain them.")
    run_registry = pd.DataFrame([
        {"run_id": f"{'lora_r8' if method == 'lora' else method}_seed{seed}_noise{noise:02d}",
         "model": method, "lora_rank": "8" if method == "lora" else "", "seed": str(seed),
         "requested_noise_percent": str(noise), "status": "planned"}
        for method in ("frozen", "lora", "full_finetuning")
        for seed in RANDOM_SEEDS for noise in NOISE_PERCENTAGES])
assert {"run_id", "model"}.issubset(run_registry.columns)
assert run_registry["run_id"].str.strip().ne("").all() and run_registry["run_id"].is_unique
run_registry = run_registry.set_index("run_id")
for rank, seed, noise in expected_run_keys:
    run_id = f"lora_r{rank}_seed{seed}_noise{noise:02d}"
    if run_id in run_registry.index:
        assert run_registry.at[run_id, "model"] == "lora", run_id

lora_test_result_rows, lora_per_class_rows = [], []
lora_test_predictions = {}
for rank in LORA_RANKS:
    for seed in RANDOM_SEEDS:
        for noise_percent in NOISE_PERCENTAGES:
            run_key = (rank, seed, noise_percent)
            label_key = (seed, noise_percent)
            synchronize_gpu()
            reset_gpu_peak_memory()
            model, metadata = restore_lora_model(run_key)
            validation_record = lora_validation_metadata[run_key]
            assert metadata["training_run_id"] == validation_record["training_run_id"] == lora_training_run_id, "Models changed since validation; rerun section 7."
            synchronize_gpu()
            test_start = time.perf_counter()
            test_metrics = evaluate_lora_model(model, "test", true_test_labels)
            synchronize_gpu()
            test_seconds = time.perf_counter() - test_start
            lora_test_predictions[run_key] = test_metrics["predictions"]
            report = classification_report(true_test_labels, test_metrics["predictions"],
                                           labels=CLASS_IDS, output_dict=True, zero_division=0)
            class_scores = [
                {"class_id": int(class_id), "class_name": str(class_names.loc[class_id]),
                 "precision": report[str(class_id)]["precision"], "recall": report[str(class_id)]["recall"],
                 "f1": report[str(class_id)]["f1-score"], "support": int(report[str(class_id)]["support"])}
                for class_id in CLASS_IDS]
            lora_per_class_rows.extend({"lora_rank": rank, "seed": seed, "noise_percent": noise_percent, **row}
                                       for row in class_scores)
            validation_row = validation_by_run.loc[run_key]
            measurements = metadata["measurements"]
            result = {
                "run_id": f"lora_r{rank}_seed{seed}_noise{noise_percent:02d}",
                "model": "lora", "lora_rank": rank, "seed": seed,
                "lora_alpha": metadata["lora_alpha"], "lora_dropout": LORA_DROPOUT,
                "lora_target_modules": json.dumps(list(LORA_TARGET_MODULES)),
                "lora_bias": "none", "lora_use_rslora": False, "lora_init_weights": True,
                "lora_task_type": "FEATURE_EXTRACTION",
                "requested_noise_percent": noise_percent,
                "actual_noise_percent": 100 * np.mean(training_label_variants[label_key] != original_training_labels),
                "status": "completed", "final_epoch": metadata["epoch"],
                "validation_accuracy": validation_row["accuracy"], "validation_macro_f1": validation_row["macro_f1"],
                "test_accuracy": test_metrics["accuracy"], "test_macro_f1": test_metrics["macro_f1"],
                "training_seconds": measurements["training_seconds"],
                "training_and_validation_seconds": measurements["training_and_validation_seconds"],
                "epoch_validation_seconds": measurements["epoch_validation_seconds"],
                "final_validation_seconds": validation_record["seconds"], "test_inference_seconds": test_seconds,
                "peak_pytorch_allocated_gib": maximum_measured_memory(measurements["peak_pytorch_allocated_gib"],
                    validation_record["peak_pytorch_allocated_gib"], peak_gpu_memory_gib()),
                "gpu_model": metadata["environment"]["gpu"], "trainable_parameters": metadata["trainable_parameters"],
                "bert_checkpoint": BERT_CHECKPOINT_NAME, "max_token_length": MAXIMUM_TOKEN_LENGTH,
                "learning_rate": LEARNING_RATE, "training_batch_size": TRAINING_BATCH_SIZE,
                "evaluation_batch_size": EVALUATION_BATCH_SIZE, "weight_decay": WEIGHT_DECAY,
                "stopping_rule": "final_epoch", "training_precision": "float32",
                "torch_version": metadata["environment"]["torch"], "transformers_version": metadata["environment"]["transformers"],
                "peft_version": metadata["environment"]["peft"],
                **{f"{split}_csv_sha256": source_hashes[split] for split in SPLIT_NAMES},
                "test_per_class_metrics_json": json.dumps(class_scores, ensure_ascii=False, allow_nan=False),
            }
            lora_test_result_rows.append(result)
            run_id = result["run_id"]
            for column in result:
                if column != "run_id" and column not in run_registry:
                    run_registry[column] = ""
            if run_id not in run_registry.index:
                run_registry.loc[run_id] = ""
            obsolete_columns = ["head_training_seconds", "test_head_inference_seconds",
                                "amortized_train_encoding_seconds", "amortized_validation_encoding_seconds", "amortized_test_encoding_seconds",
                                "validation_accuracy_drop_pp", "validation_macro_f1_drop_pp", "test_accuracy_drop_pp", "test_macro_f1_drop_pp"]
            for column in obsolete_columns:
                if column in run_registry:
                    run_registry.at[run_id, column] = ""
            for column, value in result.items():
                if column != "run_id":
                    run_registry.at[run_id, column] = "" if pd.isna(value) else str(value)
            temporary_results_path = RESULTS_PATH.with_suffix(".csv.tmp")
            run_registry.reset_index().to_csv(temporary_results_path, index=False)
            temporary_results_path.replace(RESULTS_PATH)
            print(f"Saved {run_id}.")
            del model
            gc.collect()
            if computation_device.type == "cuda":
                torch.cuda.empty_cache()

lora_final_results = pd.DataFrame(lora_test_result_rows)
lora_test_per_class = pd.DataFrame(lora_per_class_rows)
print(f"\nSaved all {len(lora_final_results)} LoRA runs to {RESULTS_PATH}.")
print("Final test metrics use a 0–1 scale:")
display(lora_final_results[["lora_rank", "seed", "requested_noise_percent", "test_accuracy", "test_macro_f1",
                            "training_seconds", "peak_pytorch_allocated_gib"]].round(4))
for rank in LORA_RANKS:
    for seed in RANDOM_SEEDS:
        for noise_percent in NOISE_PERCENTAGES:
            print(f"\nPer-class test scores: rank {rank}, seed {seed}, noise {noise_percent}%")
            scores = lora_test_per_class.loc[
                lora_test_per_class["lora_rank"].eq(rank) & lora_test_per_class["seed"].eq(seed)
                & lora_test_per_class["noise_percent"].eq(noise_percent),
                ["class_id", "class_name", "precision", "recall", "f1", "support"]]
            display(scores.reset_index(drop=True).round(4))


Saved lora_r8_seed42_noise00.
Saved lora_r8_seed42_noise10.
Saved lora_r8_seed42_noise20.
Saved lora_r8_seed42_noise40.
Saved lora_r8_seed43_noise00.
Saved lora_r8_seed43_noise10.
Saved lora_r8_seed43_noise20.
Saved lora_r8_seed43_noise40.
Saved lora_r8_seed44_noise00.
Saved lora_r8_seed44_noise10.
Saved lora_r8_seed44_noise20.
Saved lora_r8_seed44_noise40.

Saved all 12 LoRA runs to /content/run_results.csv.
Final test metrics use a 0–1 scale:


,lora_rank,seed,requested_noise_percent,test_accuracy,test_macro_f1,training_seconds,peak_pytorch_allocated_gib
0,8,42,0,0.8921,0.8840,389.7972,3.9486
1,8,42,10,0.8849,0.8768,387.8587,3.9487
2,8,42,20,0.8812,0.8720,387.9274,3.9487
3,8,42,40,0.8655,0.8493,387.4582,3.9487
4,8,43,0,0.8866,0.8741,387.3369,3.9487
5,8,43,10,0.8846,0.8701,388.0948,3.9487
6,8,43,20,0.8795,0.8659,387.0855,3.9487
7,8,43,40,0.8641,0.8443,386.9612,3.9487
8,8,44,0,0.8878,0.8777,387.6305,3.9487
9,8,44,10,0.8849,0.8719,387.6293,3.9487



Per-class test scores: rank 8, seed 42, noise 0%


,class_id,class_name,precision,recall,f1,support
0,0,Cardiovascular diseases,0.8369,0.9129,0.8732,781
1,1,Chronic kidney disease,0.8624,0.8624,0.8624,109
2,2,Chronic respiratory diseases,0.8701,0.8599,0.8650,257
3,3,Diabetes mellitus,0.9114,0.9270,0.9191,233
4,4,Digestive diseases,0.7368,0.7636,0.7500,110
5,5,HIV/AIDS,0.9470,0.9408,0.9439,152
6,6,Hepatitis A/B/C/E,0.9000,0.9000,0.9000,60
7,7,Mental disorders,0.8942,0.8795,0.8867,365
8,8,Musculoskeletal disorders,0.9397,0.9233,0.9315,287
9,9,Neoplasms (cancer),0.9404,0.9049,0.9223,820



Per-class test scores: rank 8, seed 42, noise 10%


...

,class_id,class_name,precision,recall,f1,support
0,0,Cardiovascular diseases,0.8538,0.8899,0.8715,781
1,1,Chronic kidney disease,0.8532,0.8532,0.8532,109
2,2,Chronic respiratory diseases,0.8594,0.8560,0.8577,257
3,3,Diabetes mellitus,0.9198,0.9356,0.9277,233
4,4,Digestive diseases,0.7245,0.6455,0.6827,110
5,5,HIV/AIDS,0.9527,0.9276,0.9400,152
6,6,Hepatitis A/B/C/E,0.8983,0.8833,0.8908,60
7,7,Mental disorders,0.8908,0.8712,0.8809,365
8,8,Musculoskeletal disorders,0.9255,0.9094,0.9174,287
9,9,Neoplasms (cancer),0.9161,0.9183,0.9172,820



Per-class test scores: rank 8, seed 44, noise 20%


,class_id,class_name,precision,recall,f1,support
0,0,Cardiovascular diseases,0.8579,0.8809,0.8692,781
1,1,Chronic kidney disease,0.8468,0.8624,0.8545,109
2,2,Chronic respiratory diseases,0.8725,0.8521,0.8622,257
3,3,Diabetes mellitus,0.9185,0.9185,0.9185,233
4,4,Digestive diseases,0.7037,0.6909,0.6972,110
5,5,HIV/AIDS,0.9533,0.9408,0.9470,152
6,6,Hepatitis A/B/C/E,0.9138,0.8833,0.8983,60
7,7,Mental disorders,0.8858,0.8712,0.8785,365
8,8,Musculoskeletal disorders,0.9097,0.9129,0.9113,287
9,9,Neoplasms (cancer),0.9124,0.9146,0.9135,820



Per-class test scores: rank 8, seed 44, noise 40%


,class_id,class_name,precision,recall,f1,support
0,0,Cardiovascular diseases,0.8492,0.8656,0.8573,781
1,1,Chronic kidney disease,0.8288,0.8440,0.8364,109
2,2,Chronic respiratory diseases,0.8696,0.8560,0.8627,257
3,3,Diabetes mellitus,0.9185,0.9185,0.9185,233
4,4,Digestive diseases,0.6383,0.5455,0.5882,110
5,5,HIV/AIDS,0.9459,0.9211,0.9333,152
6,6,Hepatitis A/B/C/E,0.9138,0.8833,0.8983,60
7,7,Mental disorders,0.8663,0.8521,0.8591,365
8,8,Musculoskeletal disorders,0.8768,0.8676,0.8722,287
9,9,Neoplasms (cancer),0.8882,0.9207,0.9042,820
